In [1]:
import os
import importlib

try:
  from google.colab import drive
  drive.mount('/content/drive')
  IN_COLAB = True
except:
  IN_COLAB = False

Mounted at /content/drive


In [ ]:
if IN_COLAB:
    !pip install -q "peft==0.13.2" "transformers==4.57.3" "lm_eval==0.4.9.2"

In [ ]:
BASE_RESULTS_DIR = "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments"
LOG_DIR = os.path.join(BASE_RESULTS_DIR, "logs")
MODEL_DIR = os.path.join(BASE_RESULTS_DIR, "models")
BASE_EVAL_DIR = os.path.join(BASE_RESULTS_DIR, "eval")

if IN_COLAB:
    os.makedirs(LOG_DIR, exist_ok=True)
    os.makedirs(MODEL_DIR, exist_ok=True)
    os.makedirs(BASE_EVAL_DIR, exist_ok=True)

In [ ]:
model = "Qwen/Qwen2-1.5B-Instruct" # "facebook/opt-125m", "facebook/opt-1.3b", "Qwen/Qwen2-0.5B"
model_name = model.replace("/", "-")

In [ ]:
import json
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

def build_context(example):
    paragraphs = []

    for paragraph_title, sentences in example["context"]:
        paragraph_text = " ".join(sentences)
        paragraphs.append(f"[{paragraph_title}] {paragraph_text}")

    return "\n".join(paragraphs)

def generate_predictions(model_name, dataset_path, out_path, max_new_tokens=64, limit=None):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name).cuda()
    model.eval()

    model.generation_config.pad_token_id = tokenizer.pad_token_id

    with open(dataset_path, "r", encoding="utf-8") as f:
        examples = json.load(f)

    predictions = {
        "answer": {},
        "sp": {}
    }

    if limit is None:
        limit = len(examples)

    for example in tqdm(examples[:limit]):
        question_id  = example["_id"]
        question_text = example["question"]

        context_text = build_context(example)

        prompt = (
            "Answer the question using the context. "
            "Reply only with a short answer."
            f"Context: \n{context_text}\n\n"
            f"Question: {question_text}\n\n"
            "Answer: "
        )

        model_inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=getattr(model.config, "max_position_embeddings", None),
        ).to(model.device)

        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

        generated_text = tokenizer.decode(
            generated_ids[0][model_inputs["input_ids"].shape[-1]:],
            skip_special_tokens=True
        )

        short_answer = generated_text.splitlines()[0].strip()

        predictions["answer"][question_id] = short_answer
        predictions["sp"][question_id] = []

    with open(out_path, "w", encoding="utf-8") as f:
          json.dump(predictions, f)

    print(f"Predictions saved to {out_path}")

In [ ]:
OUT_DIR = os.path.join(BASE_EVAL_DIR, f"{model_name}_hotpot_predictions")
os.makedirs(OUT_DIR, exist_ok=True)

generate_predictions(
    model,
    "data/hotpot_dev_distractor_v1.json", #the file has to be stored in this path
    os.path.join(OUT_DIR, f"hotpot_predictions_{model_name}.json"),
    limit=None)